In [1]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


In [3]:
from pyspark.sql import SparkSession

# Membuat SparkSession baru untuk Tugas Mandiri
spark = SparkSession.builder.appName("TugasMandiri4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Membaca data langsung dari HDFS
df_tugas = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv", header=True, inferSchema=True)

# Mengecek skema, jumlah baris, dan 10 baris pertama
df_tugas.printSchema()
print("Total baris data Tugas:", df_tugas.count())
df_tugas.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Total baris data Tugas: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semaran

In [4]:
# Eksplorasi Mandiri 1: Melihat daftar kota yang unik tanpa duplikasi
print("Daftar Kota yang Tersedia:")
df_tugas.select("kota").distinct().show()

# Eksplorasi Mandiri 2: Menampilkan 5 baris data secara acak (bukan cuma yang paling atas)
print("Sampel Data Acak:")
df_tugas.sample(fraction=0.01, seed=42).show(5)

Daftar Kota yang Tersedia:
+----------+
|      kota|
+----------+
|  Magelang|
|  Semarang|
|   Kebumen|
|      Solo|
| Purworejo|
|Yogyakarta|
+----------+

Sampel Data Acak:
+--------+-------------------+------------+---------+------------+------------+-----------------+------+
|order_id|            tanggal|    kategori|     kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+------------+---------+------------+------------+-----------------+------+
|ORD-3255|2026-09-01 00:00:00|  Elektronik| Magelang|           3|       60000|     Kartu Kredit|   2.0|
|ORD-3313|2026-09-03 00:00:00|Rumah Tangga| Semarang|          10|      350000|              COD|   5.0|
|ORD-3395|2026-09-20 00:00:00|  Elektronik|Purworejo|           1|      200000|         E-Wallet|   4.0|
|ORD-3712|2026-09-19 00:00:00|  Elektronik|     Solo|           3|      350000|    Transfer Bank|   4.0|
+--------+-------------------+------------+---------+------------+------------+----------

In [5]:
from pyspark.sql.functions import col

# Menghitung jumlah nilai kosong pada kolom rating
jumlah_kosong = df_tugas.filter(col("rating").isNull()).count()
print(f"Jumlah baris dengan rating kosong: {jumlah_kosong}")

# Menangani data kosong dengan mengisinya menggunakan angka 0
df_tugas_bersih = df_tugas.na.fill(0, subset=["rating"])

# Memverifikasi bahwa data kosong sudah hilang
sisa_kosong = df_tugas_bersih.filter(col("rating").isNull()).count()
print(f"Sisa data kosong setelah ditangani: {sisa_kosong}")

Jumlah baris dengan rating kosong: 204
Sisa data kosong setelah ditangani: 0


In [6]:
from pyspark.sql.functions import col, when

# Menambahkan kolom total_pendapatan dan tier_transaksi secara berurutan
df_tugas_transform = df_tugas_bersih.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
                                    .withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

# Menampilkan hasil transformasi untuk memverifikasi
df_tugas_transform.select("order_id", "total_pendapatan", "tier_transaksi").show(5)

+--------+----------------+--------------+
|order_id|total_pendapatan|tier_transaksi|
+--------+----------------+--------------+
|ORD-3000|          270000|         Kecil|
|ORD-3001|          600000|         Besar|
|ORD-3002|          480000|         Kecil|
|ORD-3003|         2100000|         Besar|
|ORD-3004|          600000|         Besar|
+--------+----------------+--------------+
only showing top 5 rows



In [7]:
from pyspark.sql.functions import sum as spark_sum, count, avg

print("1. Kategori dengan total pendapatan tertinggi:")
df_tugas_transform.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("total")) \
    .orderBy(col("total").desc()) \
    .show(1)

print("2. Kota dengan jumlah transaksi tier 'Besar' terbanyak:")
df_tugas_transform.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .agg(count("order_id").alias("jumlah_transaksi_besar")) \
    .orderBy(col("jumlah_transaksi_besar").desc()) \
    .show(1)

print("3. Rata-rata rating untuk masing-masing metode pembayaran:")
df_tugas_transform.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .orderBy(col("rata_rata_rating").desc()) \
    .show()

1. Kategori dengan total pendapatan tertinggi:
+------------+---------+
|    kategori|    total|
+------------+---------+
|Rumah Tangga|138665000|
+------------+---------+
only showing top 1 row

2. Kota dengan jumlah transaksi tier 'Besar' terbanyak:
+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|                    92|
+----+----------------------+
only showing top 1 row

3. Rata-rata rating untuk masing-masing metode pembayaran:
+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|         E-Wallet|             3.292|
|     Kartu Kredit|3.1910569105691056|
+-----------------+------------------+



In [8]:
# Menyimpan DataFrame hasil olahan ke direktori HDFS
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi_september"
df_tugas_transform.write.csv(output_path, header=True, mode="overwrite")

# Verifikasi apakah direktori dan partisi berkas berhasil dibuat di HDFS
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_transaksi_september

Found 2 items
-rw-r--r--   3 maulzz supergroup          0 2026-09-12 13:10 /user/mahasiswa/tugas4/hasil_transaksi_september/_SUCCESS
-rw-r--r--   3 maulzz supergroup      97296 2026-09-12 13:10 /user/mahasiswa/tugas4/hasil_transaksi_september/part-00000-b4d2477f-16ad-4a59-8952-a8fae8b3e06c-c000.csv
